# Can a 20-Year-Old Algorithm Beat a 3B LLM at Spotting Churn Signals?

**Spoiler: neither wins — and that's the interesting part.**

This notebook walks through the [`cfpb-churn-signal-golden-set`](https://www.kaggle.com/datasets/oguzkaanmavice/cfpb-churn-signal-golden-set) dataset: 1,001 real CFPB consumer complaint narratives, each hand-verified for an explicit **churn signal** — the consumer states they closed/are closing the account, switched/are switching provider, or voice a definite intention or threat to leave. This is detection (*does the text say it*), not prediction (*will they actually leave*).

**Labeling protocol, in one paragraph:** Claude proposed a label + evidence quote for every narrative, then a human reviewed and either confirmed or overrode every single row (6/1001 overrides). This two-step process exists so evaluating an LLM classifier against these labels isn't circular — the classifier scored below (`llama3.2:3b`) is a different model from a different company than the one that proposed labels.

**This dataset is for evaluation, not training.** All four CSVs below are already committed results — nothing in this notebook calls an LLM or an API. TF-IDF trains live (it's cheap); the LLM's predictions were generated once, offline, and are read straight from CSV.

In [ ]:
import pandas as pd

DATA = "/kaggle/input/cfpb-churn-signal-golden-set"

golden = pd.read_csv(f"{DATA}/golden_set.csv")
llm_preds = pd.read_csv(f"{DATA}/llm_predictions.csv")
tfidf_preds = pd.read_csv(f"{DATA}/tfidf_predictions.csv")
folds = pd.read_csv(f"{DATA}/cv_folds.csv")

print(golden.shape, llm_preds.shape, tfidf_preds.shape, folds.shape)
golden.head(3)

## Label distribution

48 of 1,001 rows (4.8%) are positive. One product structurally can't carry a churn signal: **Debt collection is 0/167 positive** — you never chose your debt collector, so there's no relationship to voluntarily end. Sampling was deliberately capped there instead of grown until noise produced a positive.

In [ ]:
import matplotlib.pyplot as plt

by_product = golden.groupby("product")["churn_signal"].agg(["count", "sum"])
by_product["rate"] = (by_product["sum"] / by_product["count"] * 100).round(1)
print(by_product)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_product["sum"].plot(kind="bar", ax=axes[0], color=["#c96f4a", "#c96f4a", "#7a8b99"])
axes[0].set_title("Positive churn-signal rows by product")
axes[0].set_ylabel("count")
golden["narrative"].str.len().hist(bins=40, ax=axes[1], color="#4a6c96")
axes[1].set_title("Narrative length (characters)")
plt.tight_layout()
plt.show()

## Why 5-fold *stratified* cross-validation, not one train/test split

With only 48 positives in 1,001 rows, a single 70/30 split holds out roughly ten positive examples. One misclassified example swings recall by about ten points — a single number from one split is closer to a coin flip than a measurement. 5-fold stratified CV scores every row exactly once as a held-out case across the 5 folds, so every one of the 48 positives contributes, and the fold-to-fold spread (reported as mean ± std) shows how much the result actually varies instead of hiding it behind one split.

Both prediction files in this dataset share the *same* canonical fold assignment (`cv_folds.csv`), so the TF-IDF model trained live below and the LLM's pre-generated predictions are scored on identical partitions — a fair, apples-to-apples comparison.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

df = golden.merge(folds, on="complaint_id")

def score(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

tfidf_fold_metrics = []
for fold_idx in range(5):
    train_df = df[df["fold"] != fold_idx]
    test_df = df[df["fold"] == fold_idx]
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english")
    X_train = vec.fit_transform(train_df["narrative"])
    X_test = vec.transform(test_df["narrative"])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    clf.fit(X_train, train_df["churn_signal"])
    y_pred = clf.predict(X_test)
    m = score(test_df["churn_signal"], y_pred)
    m["fold"] = fold_idx
    m["n_test"] = len(test_df)
    m["n_positive_test"] = int(test_df["churn_signal"].sum())
    tfidf_fold_metrics.append(m)

tfidf_metrics_df = pd.DataFrame(tfidf_fold_metrics)
tfidf_metrics_df

In [ ]:
tfidf_summary = tfidf_metrics_df[["accuracy", "precision", "recall", "f1"]].agg(["mean", "std"]).round(3)
print("TF-IDF + Logistic Regression, 5-fold CV:")
tfidf_summary

## Per-fold F1: watch the TF-IDF bar hit zero

This is the concrete argument for cross-validation over a single split: one of TF-IDF's five folds scores **F1 = 0.000**. If that fold had been the one-and-only train/test split for this project, the honest headline would have been "TF-IDF is useless here" — a different split could just as easily have looked great. Neither would have been true; only the full spread is.

In [ ]:
llm_df = llm_preds.copy()
llm_fold_metrics = []
for fold_idx in range(5):
    fold_df = llm_df[llm_df["fold"] == fold_idx]
    m = score(fold_df["true_label"], fold_df["predicted_label"])
    m["fold"] = fold_idx
    llm_fold_metrics.append(m)
llm_metrics_df = pd.DataFrame(llm_fold_metrics)

fig, ax = plt.subplots(figsize=(9, 4))
width = 0.35
x = np.arange(5)
ax.bar(x - width / 2, tfidf_metrics_df["f1"], width, label="TF-IDF + LR", color="#7a8b99")
ax.bar(x + width / 2, llm_metrics_df["f1"], width, label="LLM zero-shot", color="#c96f4a")
ax.set_xticks(x)
ax.set_xticklabels([f"fold {i}" for i in range(5)])
ax.set_ylabel("F1")
ax.set_title("Per-fold F1 — same 5 folds, two models")
ax.legend()
plt.tight_layout()
plt.show()

llm_summary = llm_metrics_df[["accuracy", "precision", "recall", "f1"]].agg(["mean", "std"]).round(3)
print("LLM zero-shot (llama3.2:3b), 5-fold CV:")
llm_summary

In [ ]:
tfidf_all = df.merge(tfidf_preds[["complaint_id", "predicted_label"]], on="complaint_id")
cm_tfidf = confusion_matrix(tfidf_all["churn_signal"], tfidf_all["predicted_label"], labels=[0, 1])
cm_llm = confusion_matrix(llm_df["true_label"], llm_df["predicted_label"], labels=[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm, title in zip(axes, [cm_tfidf, cm_llm], ["TF-IDF + LR", "LLM zero-shot"]):
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["pred 0", "pred 1"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["actual 0", "actual 1"])
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.tight_layout()
plt.show()

print("TF-IDF confusion matrix:\n", cm_tfidf)
print("LLM confusion matrix:\n", cm_llm)

## Reading the trade-off

Across all 1,001 held-out predictions: TF-IDF caught **10 of 48** real churn signals and raised 13 false alarms. The LLM caught **31 of 48** — three times as many — and raised 112 false alarms doing it.

Don't be fooled by accuracy. On a 95.2%-negative dataset, high accuracy is what you get for saying "no" a lot; TF-IDF's ~0.95 accuracy is mostly that arithmetic. The rows that actually reflect the task are **recall** and **F1**. There the trade is stark: TF-IDF is precise but mostly blind (recall 0.213); the LLM sees roughly two-thirds of real signals at the cost of far more false positives (recall 0.647).

Notice the standard deviations too: the LLM isn't just higher-recall, it's the *more stable* estimator — its metrics vary ±0.02–0.04 across folds, while TF-IDF's precision swings ±0.264 depending on which ten positives happen to land in the test fold.

## Error analysis: what does an LLM false positive actually look like?

The LLM's structured-output schema generates the boolean verdict *before* the rationale — field order is generation order — so some rationales below may be post-hoc justifications of an already-committed answer rather than the reasoning that produced it.

In [ ]:
false_positives = llm_df[(llm_df["true_label"] == 0) & (llm_df["predicted_label"] == 1)]
print(f"{len(false_positives)} false positives out of {len(llm_df)} rows\n")
for _, row in false_positives.head(5).iterrows():
    print(f"complaint_id={row['complaint_id']}")
    print(f"  rationale: {row['rationale'][:200]}")
    print()

## The price tag

Numbers below are cited from this project's committed cost-comparison measurement (`results_cost_comparison.json` in the [GitHub repo](https://github.com/EnigmaDevelop/ai-engineering/tree/main/chapters/03-eval-harness)), not recomputed here — Kaggle's shared CPU isn't a fair benchmark for the LLM side, which was timed once on a fixed reference machine.

| | TF-IDF + LR | LLM zero-shot |
|---|---|---|
| Throughput | 4,943 rows/s (full fit + predict) | 0.333 rows/s |
| Per-row cost | ~0.2 ms | ~3.0 s (± 1.4 s) |
| Tokens per call | n/a | ~393 prompt + ~33 response (sampled, n=30) |

**TF-IDF is roughly 14,844× faster per row.** So the honest engineering answer isn't "the LLM has better F1, ship it" — it's a routing question. If flagged complaints go to a human for review, the LLM's 3× recall is probably worth its false-positive rate and its cost. If the label triggers automatic action, or you're scoring millions of rows, the 20-year-old algorithm is still the right default.

In [ ]:
costs = pd.Series({"TF-IDF + LR": 4943.0, "LLM zero-shot": 0.333}, name="rows/s")
ax = costs.plot(kind="bar", logy=True, color=["#7a8b99", "#c96f4a"], figsize=(5, 4))
ax.set_ylabel("rows/s (log scale)")
ax.set_title("Throughput: ~14,844x gap")
plt.tight_layout()
plt.show()

## Takeaways

1. **The golden set is the asset; the models are interchangeable.** Any future classifier, prompting technique, or judge can be measured against these same 1,001 human-verified rows and the same 5 folds.
2. **On a 4.8%-positive class, a single train/test split is a coin flip wearing a lab coat.** This notebook's own TF-IDF run shows per-fold F1 ranging from 0.000 to 0.462 on the *same data* — report mean ± std across folds, or accept your number is partly luck.
3. **Neither model won, and that's a publishable result.** TF-IDF: 10/48 signals caught, 13 false alarms, ~0.2 ms/row. LLM: 31/48 caught, 112 false alarms, ~3 s and ~426 tokens/row. The right choice depends on what's downstream of the label.

**Links:** [dataset card](https://www.kaggle.com/datasets/oguzkaanmavice/cfpb-churn-signal-golden-set) · [full methodology on GitHub](https://github.com/EnigmaDevelop/ai-engineering/tree/main/chapters/03-eval-harness) (labeling rubric, edge-case taxonomy, LLM-as-judge bias measurements).